In [23]:
import os
import shutil
import glob

WORKING = "/kaggle/working"
os.chdir(WORKING)

# Créer les dossiers cibles (avec les bons noms, sans accent)
for d in ["sources", "referentiels", "sortie"]:
    os.makedirs(d, exist_ok=True)

# Mapping : nom canonique attendu -> variantes possibles dans le dataset
SOURCES = {
    "plateforme_immo_sn.csv": ["plateforme_immo_sn.csv", "plate-forme_immo_sn.csv", "plate_forme_immo_sn.csv"],
    "agence_teranga.xlsx":    ["agence_teranga.xlsx", "agency_teranga.xlsx"],
    "particuliers.json":      ["particuliers.json", "particulier.json"],
    "notaires.csv":           ["notaires.csv"],
    "import_historique.csv":  ["import_historique.csv"],
}

REFERENTIELS = {
    "referentiel_geographique.csv": ["referentiel_geographique.csv"],
    "referentiel_categories.csv":   ["referentiel_categories.csv", "référence_categories.csv", "reference_categories.csv"],
    "dictionnaire_donnees.csv":     ["dictionnaire_donnees.csv"],
    "regles_metier.md":             ["regles_metier.md"],
}

def trouver_et_copier(canon, variantes, dossier_cible):
    """Cherche le fichier n'importe où sous /kaggle/input/ et le copie au nom canonique."""
    cible = os.path.join(dossier_cible, canon)
    if os.path.exists(cible):
        print(f"OK (déjà présent) : {cible}")
        return True
    for variante in variantes:
        matches = glob.glob(f"/kaggle/input/**/{variante}", recursive=True)
        if matches:
            shutil.copy(matches[0], cible)
            print(f"COPIÉ : {matches[0]} -> {cible}")
            return True
    print(f"INTROUVABLE : {canon} (variantes testées : {variantes})")
    return False

print("=== Recopie vers noms canoniques ===")
for canon, variantes in SOURCES.items():
    trouver_et_copier(canon, variantes, "sources")
for canon, variantes in REFERENTIELS.items():
    trouver_et_copier(canon, variantes, "referentiels")

# Nettoyer les éventuels dossiers mal nommés (réfentiels, etc.)
for d in os.listdir("."):
    if os.path.isdir(d) and d not in ("sources", "referentiels", "sortie") and not d.startswith("."):
        contenu = os.listdir(d)
        if any("referentiel" in f or "regles" in f for f in contenu):
            print(f"Suppression du dossier mal nommé : {d}/")
            shutil.rmtree(d)

# Vérification finale
print("\n=== VÉRIFICATION FINALE ===")
print(f"sources/      : {sorted(os.listdir('sources'))}")
print(f"referentiels/ : {sorted(os.listdir('referentiels'))}")

=== Recopie vers noms canoniques ===
OK (déjà présent) : sources/plateforme_immo_sn.csv
OK (déjà présent) : sources/agence_teranga.xlsx
OK (déjà présent) : sources/particuliers.json
OK (déjà présent) : sources/notaires.csv
OK (déjà présent) : sources/import_historique.csv
OK (déjà présent) : referentiels/referentiel_geographique.csv
OK (déjà présent) : referentiels/referentiel_categories.csv
OK (déjà présent) : referentiels/dictionnaire_donnees.csv
OK (déjà présent) : referentiels/regles_metier.md

=== VÉRIFICATION FINALE ===
sources/      : ['agence_teranga.xlsx', 'import_historique.csv', 'notaires.csv', 'particuliers.json', 'plateforme_immo_sn.csv']
referentiels/ : ['dictionnaire_donnees.csv', 'referentiel_categories.csv', 'referentiel_geographique.csv', 'regles_metier.md']


# Étape 1 et 2 — Socle Python

## Ce que fait cette cellule

Elle définit tout ce qui sera utilisé dans les cellules suivantes :

1. **Classes métier**
   - `BienImmobilier` : représente un bien après harmonisation. Toutes les sources (Excel, CSV, JSON) seront ramenées à ce schéma unique.
   - `RegleValidation` : encapsule une règle métier (nom, gravité, fonction de test).

2. **Exceptions personnalisées**
   - `ErreurValidation` : anomalie bloquante, la ligne ira en quarantaine.
   - `AvertissementValidation` : anomalie non bloquante, la ligne est conservée avec signalement.

3. **Utilitaires**
   - `corriger_mojibake` : corrige les textes mal encodés (`ThiÃ¨s` → `Thiès`). La correction complète de `U+FFFD` (information perdue) se fera en étape 3 avec le référentiel géographique.
   - `normaliser_casse` : uniformise la casse (`DAKAR`, `Dakar`, `dakar` → `Dakar`).
   - `parser_date` : convertit une date en objet `datetime`, gère ISO, JJ/MM/AAAA, JJ-MM-AAAA.
   - `parser_prix` : convertit un prix en `float`, gère `26 545 000`, `147.52 millions`, `1,250` (ambigu), `à négocier`.

## Justifications des choix

| Choix | Justification CDC |
|---|---|
| Classes pour représenter les données | Étape 2 du CDC : « Utiliser une classe simple pour représenter un bien ou une règle » |
| 2 exceptions distinctes | Section 6.1 : distinguer erreur bloquante et avertissement |
| `parser_prix` retourne un tuple (valeur, motif) | Section 7 : chaque conversion doit produire un indicateur chiffré |
| Ambiguité `1,250` traitée en séparateur de milliers | Section 6.3 : « 1,25 m² invraisemblable, 1 250 m² plausible » |
| `date future` rejetée | Section 6.1 : quarantaine si date future |

## Ce que tu dois voir à la fin

À l'exécution, la cellule affiche :

\`\`\`
Socle Python OK : classes, exceptions et utilitaires validés.
\`\`\`

Si un `AssertionError` apparaît, c'est un bug dans le code : envoie-moi le message.

## Limites assumées

- **Mojibake `U+FFFD`** : le caractère `ï¿½` correspond à une information irrécupérable (elle a été remplacée lors d'un mauvais décodage antérieur). La résolution se fera par matching avec le référentiel géographique en étape 3.
- **Prix `1,250`** : traité comme 1 250 (séparateur de milliers). Cette décision est documentée et sera vérifiée par source en étape 3.
- **Dates relatives** (`hier`) : rejetées faute de contexte exploitable, elles iront en quarantaine ou en avertissement selon le champ.

In [ ]:
from datetime import datetime
import re

# =============================================================
# 1. EXCEPTIONS PERSONNALISÉES (étape 1 du CDC)

class ErreurValidation(Exception):
    """Anomalie bloquante : la ligne ira en quarantaine."""
    pass

class AvertissementValidation(Exception):
    """Anomalie non bloquante : la ligne est conservée avec signalement."""
    pass

# =============================================================
# 2. CLASSE BIEN IMMOBILIER (étape 2 : POO)

class BienImmobilier:
    """
    Représente un bien immobilier unifié, issu d'une des 5 sources.
    Toutes les sources sont ramenées à ce schéma cible unique.
    """
 
    CHAMPS = [
        "identifiant_local", "source", "date_mise_en_ligne",
        "type_transaction", "periodicite", "categorie",
        "region", "commune", "prix", "surface",
        "chambres", "salles_bain", "statut", "quartier", "description"
    ]

    def __init__(self, **kwargs):
        for champ in self.CHAMPS:
            setattr(self, champ, kwargs.get(champ))
        self.motif_quarantaine = None
        self.avertissements = []

    def to_dict(self):
        return {c: getattr(self, c) for c in self.CHAMPS}

    def __repr__(self):
        return f"<Bien {self.identifiant_local} | {self.source} | {self.categorie} | {self.prix}>"
            ""README.md : 17 78 octets
            antimants.csv : 26666 octets
            donnees_propress.csv : 334071 octets
            donnees_propres.json : 1212634 octets
            quarantine.csv : 28916 octets
            rapport_qualite.csv : 1751 octets
            rapport_qualite.json : 7936 octets

            === Résumé final ===
            Biens bruts chargés : 3184
            Biens propres : 2811
            Biens avec avertissement: 168
            Biens en quarantaine : 205
            Dont doubles exacts : 15
            Taux de conformité : 88,29 %

            Pour récupérer : onglet 'Output' à droite → dossier 'sortie/'
            ttr(self, champ, kwargs.get(champ))
        self.motif_quarantaine = None
        self.avertissements = []

    def to_dict(self):
        return {c: getattr(self, c) for c in self.CHAMPS}

    def __repr__(self):
        return f"<Bien {self.identifiant_local} | {self.source} | {self.categorie} | {self.prix}>"

# =============================================================
# 3. CLASSE RÈGLE DE VALIDATION (étape 2 : POO)

class RegleValidation:
    """
    Encapsule une règle métier : nom, gravité, fonction de test.
    Permet de séparer les règles du code de traitement (CDC section 6).
    """

    GRAVITES = {"bloquante", "avertissement", "normalisable"}

    def __init__(self, nom, gravite, fonction, description=""):
        assert gravite in self.GRAVITES, f"Gravité invalide : {gravite}"
        self.nom = nom
        self.gravite = gravite
        self.fonction = fonction
        self.description = description

    def evaluer(self, bien):
        """Retourne (bool conforme, message)."""
        try:
            return self.fonction(bien), ""
        except ErreurValidation as e:
            return False, str(e)
        except AvertissementValidation as e:
            return True, str(e)
        except Exception as e:
            return False, f"Erreur inattendue : {e}"

# =============================================================
# 4. UTILITAIRES DE NORMALISATION

def corriger_mojibake(s):
    """
    Corrige le double-encodage classique (ex: 'ThiÃ¨s' -> 'Thiès').
    Ne peut PAS corriger 'ï¿½' (U+FFFD) car l'information est perdue.
    La résolution des U+FFFD se fera en étape 3 via le référentiel géographique.
    """
    if not isinstance(s, str):
        return s
    try:
        return s.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


def normaliser_casse(s):
    """Normalise une chaîne : strip + title case avec gestion des tirets."""
    if not isinstance(s, str):
        return s
    s = s.strip()
    return "-".join(p.capitalize() for p in s.split("-"))


def parser_date(texte):
    """
    Tente plusieurs formats de date.
    Retourne (datetime ou None, motif).
    Formats couverts : ISO, JJ/MM/AAAA, JJ-MM-AAAA, AAAA/MM/JJ.
    """
    if texte is None or texte == "":
        return None, "date vide"
    if isinstance(texte, datetime):
        return texte, None
    if not isinstance(texte, str):
        return None, f"type de date inattendu : {type(texte).__name__}"
    texte = texte.strip()
    if texte.lower() == "hier":
        return None, "date relative non interprétable sans contexte"
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(texte, fmt), None
        except ValueError:
            continue
    return None, f"format de date inconnu : {texte}"


def parser_prix(texte, periodicite=None):
    """
    Convertit un prix en float. Gère :
    - '551163000' -> 551163000.0
    - '26 545 000' -> 26545000.0
    - '147.52 millions' -> 147520000.0
    - '65575000 FCFA' -> 65575000.0
    - '58.29 millions' -> 58290000.0
    - '1,250' -> 1250.0 (séparateur de milliers, cf CDC 6.3)
    - '367,4' -> 367.4 (virgule décimale)
    - 'à négocier' -> (None, motif)
    """
    if texte is None:
        return None, "prix vide"
    if isinstance(texte, (int, float)):
        return float(texte), None
    if not isinstance(texte, str):
        return None, f"type de prix inattendu : {type(texte).__name__}"

    s = texte.strip().lower()

    if s in ("à négocier", "a negocier", "none fcfa", ""):
        return None, f"prix non numérique : '{texte}'"

    # Retirer les suffixes monétaires
    s = re.sub(r"\s*(fcfa|fcfa/mois|fcfa\s*/\s*mois)\s*$", "", s).strip()

    # Cas 'X millions'
    match_millions = re.match(r"^([\d\.,]+)\s*millions?$", s)
    if match_millions:
        valeur = match_millions.group(1).replace(",", ".")
        try:
            return float(valeur) * 1_000_000, None
        except ValueError:
            return None, f"prix millions illisible : '{texte}'"

    # Nettoyer les espaces de séparation de milliers
    s = s.replace(" ", "")

    # Ambiguité : '1,250' -> 1250 (séparateur) ; '367,4' -> 367.4 (décimal)
    if re.match(r"^\d+,\d{3}$", s):
        s = s.replace(",", "")
    else:
        s = s.replace(",", ".")

    try:
        return float(s), None
    except ValueError:
        return None, f"prix non convertible : '{texte}'"

# =============================================================
# 5. SELF-TEST (assertions qui passent)


# Cas classiques
assert parser_prix("26 545 000")[0] == 26545000.0
assert parser_prix("147.52 millions")[0] == 147520000.0
assert parser_prix("58.29 millions")[0] == 58290000.0
assert parser_prix("551163000")[0] == 551163000.0
assert parser_prix("65575000 FCFA")[0] == 65575000.0
assert parser_prix("1,250")[0] == 1250.0
assert parser_prix("367,4")[0] == 367.4

# Cas rejetés (valeur None + motif non vide)
assert parser_prix("à négocier")[0] is None
assert parser_prix("None FCFA")[0] is None
assert parser_prix("")[0] is None

# Dates
assert parser_date("2024-03-01")[0] is not None
assert parser_date("01/03/2024")[0] is not None
assert parser_date("32/13/2024")[0] is None
assert parser_date("hier")[0] is None
assert parser_date("")[0] is None

# Normalisation casse
assert normaliser_casse("DAKAR") == "Dakar"
assert normaliser_casse("dakar") == "Dakar"
assert normaliser_casse("Saint-Louis") == "Saint-Louis"
assert normaliser_casse("  thies  ") == "Thies"

# Mojibake : double-encodage
assert corriger_mojibake("ThiÃ¨s") == "Thiès"

# POO
b = BienImmobilier(identifiant_local="TEST-001", source="agence", prix=100000)
assert b.identifiant_local == "TEST-001"
assert b.surface is None
assert b.to_dict()["prix"] == 100000

print("Socle Python OK : classes, exceptions et utilitaires validés.")

Socle Python OK : classes, exceptions et utilitaires validés.


# Étape 3 — Chargement des 5 sources

## Ce que fait cette cellule

Elle lit les 5 sources hétérogènes et les ramène toutes à un schéma unique (`BienImmobilier`). Elle produit :

- `biens_bruts` : liste de `BienImmobilier` (toutes sources confondues)
- `inventaire_initial` : comptage par source (lignes lues, retenues, erreurs de parsing)

## Traitements spécifiques par source

| Source | Traitement |
|---|---|
| `plateforme_immo_sn.csv` | Filtrer le header dupliqué en cours de fichier. Résoudre les prix `"147.52 millions"`, `"à négocier"`. Résoudre `"appart"` → `Appartement` via le référentiel des synonymes. |
| `agence_teranga.xlsx` | Ne lire que la feuille `Annonces`, ignorer `Params` (métadonnées). Convertir `Date` (déjà `datetime` via openpyxl). |
| `particuliers.json` | Aplatir le JSON imbriqué (`localisation.region`, `details.prix`). |
| `notaires.csv` | Uniformiser les statuts (`VENDU`, `Vente conclue`, `Acte signé` → `Vendu`). |
| `import_historique.csv` | Corriger le mojibake par fuzzy matching avec le référentiel géographique (les `ï¿½` sont des caractères U+FFFD irrécupérables, seul le rapprochement avec le référentiel fonctionne). Traiter `N/A` comme `None`. |

## Justifications des choix

| Choix | Justification CDC |
|---|---|
| Ajout d'une colonne `source` sur chaque bien | Section 5.3 : « Aucun identifiant n'est universel », il faut tracer la provenance. |
| Ignorer la feuille `Params` | Section 5.1 : « Excel, 2 feuilles ». Seule `Annonces` contient les données. |
| Résolution mojibake par fuzzy matching | Les `ï¿½` sont des U+FFFD (information perdue). Seule la comparaison avec le `referentiel_geographique.csv` permet de retrouver la ville. |
| Synonymes via `referentiel_categories.csv` | Section 5.5 : « fournit les synonymes admis ». |
| Normalisation des statuts notaires | Sinon 4 modalités pour la même information, ce qui fausse la comp 3. |
| Tuples `(valeur, motif)` pour `parser_prix` et `parser_date` | Traçabilité : chaque conversion ratée alimente un avertissement. |

## Détection du mojibake

Le fichier `import_historique.csv` contient des séquences `ï¿½` (3 caractères) qui sont l'affichage de l'octet U+FFFD encodé en UTF-8. L'information est **perdue** : `Thiï¿½s` ne peut pas redevenir `Thiès` par simple ré-encodage.

La cellule résout le problème en deux temps :
1. Tentative de double-décodage (`corriger_mojibake`), qui marche pour `ThiÃ¨s` → `Thiès`.
2. Pour les cas irrécupérables (`ï¿½`), **fuzzy matching** avec le `referentiel_geographique.csv` via `difflib.SequenceMatcher` (seuil 0.75).

## Ce que tu dois voir à la fin


In [ ]:
import csv
import json
import io
import os
import unicodedata
from difflib import SequenceMatcher
import openpyxl

os.chdir("/kaggle/working")

# =============================================================
# RÉFÉRENTIELS (chargés une fois pour la résolution)

def resoudre_categorie(brut, mapping_synonymes):
    """Résout une catégorie via le mapping synonyme -> canonique."""
    if brut is None:
        return None
    cle = normaliser_pour_matching(brut)
    if not cle:
        return None
    return mapping_synonymes.get(cle, brut.strip() if isinstance(brut, str) else brut)

print("resoudre_categorie définie.")


def charger_synonymes_categories(chemin):
    """Charge le mapping synonyme -> catégorie canonique depuis referentiel_categories.csv."""
    mapping = {}
    with open(chemin, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            canonique = row["categorie"].strip()
            mapping[normaliser_pour_matching(canonique)] = canonique
            for syn in row.get("synonymes", "").split("|"):
                syn = syn.strip()
                if syn:
                    mapping[normaliser_pour_matching(syn)] = canonique
    return mapping


def charger_referentiel_geo(chemin):
    """Charge {region: set(villes)} depuis referentiel_geographique.csv."""
    ref = {}
    with open(chemin, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            region = normaliser_casse(row["region"])
            ville = normaliser_casse(row["ville"])
            ref.setdefault(region, set()).add(ville)
    return ref


def normaliser_pour_matching(s):
    """Retire les accents et ne garde que a-z0-9, pour matching approximatif."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]", "", s.lower())


def resoudre_par_referentiel(brut, referentiel, seuil=0.75):
    """
    Cherche la meilleure correspondance entre `brut` et les valeurs de `referentiel`.
    Utilise SequenceMatcher (difflib, stdlib).
    Retourne la valeur canonique ou None si aucun match au-dessus du seuil.
    """
    if not brut:
        return None
    brut_norm = normaliser_pour_matching(brut)
    if not brut_norm:
        return None
    best, best_ratio = None, 0.0
    for valide in referentiel:
        valide_norm = normaliser_pour_matching(valide)
        if not valide_norm:
            continue
        ratio = SequenceMatcher(None, brut_norm, valide_norm).ratio()
        if ratio > best_ratio:
            best, best_ratio = valide, ratio
    return best if best_ratio >= seuil else None


def parser_int(texte):
    """Convertit en int, retourne None si non convertible."""
    if texte is None:
        return None
    if isinstance(texte, bool):
        return None
    if isinstance(texte, (int, float)):
        return int(texte)
    if isinstance(texte, str):
        s = texte.strip()
        if s.isdigit():
            return int(s)
    return None


def lire_csv_avec_fallback(chemin, encodages=("utf-8", "latin-1", "cp1252")):
    """Essaie plusieurs encodages pour lire un CSV, retourne (contenu, encodage_utilisé)."""
    for enc in encodages:
        try:
            with open(chemin, encoding=enc) as f:
                return f.read(), enc
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Aucun encodage n'a fonctionné pour {chemin}")


# Chargement des référentiels (une seule fois)
SYNONYMES_CATEGORIES = charger_synonymes_categories("referentiels/referentiel_categories.csv")
REFERENTIEL_GEO = charger_referentiel_geo("referentiels/referentiel_geographique.csv")

# =============================================================
# CHARGEURS PAR SOURCE

biens_bruts = []
inventaire_initial = {}


def charger_plateforme(chemin):
    biens, erreurs = [], []
    contenu, enc = lire_csv_avec_fallback(chemin)
    reader = csv.DictReader(io.StringIO(contenu), delimiter=";")
    for i, row in enumerate(reader, start=1):
        # Filtrer le header dupliqué
        if row.get("ref_plateforme", "").strip() == "ref_plateforme":
            continue
        identifiant = (row.get("ref_plateforme") or "").strip()
        if not identifiant:
            erreurs.append(f"ligne {i} : identifiant absent")
            continue

        prix, motif_prix = parser_prix(row.get("prix", ""))
        surface, _ = parser_prix(row.get("surface", ""))
        date, motif_date = parser_date(row.get("date_mise_en_ligne"))

        # Résoudre la catégorie via synonymes
        categorie = resoudre_categorie(row.get("categorie", ""), SYNONYMES_CATEGORIES)

        # Périodicité : détecter via devise 'FCFA/mois'
        devise = (row.get("devise") or "").lower()
        periodicite = "mensuel" if "/mois" in devise else None

        bien = BienImmobilier(
            identifiant_local=identifiant,
            source="plateforme",
            date_mise_en_ligne=date,
            type_transaction=(row.get("type_transaction") or "").strip().lower() or None,
            periodicite=periodicite,
            categorie=categorie,
            region=normaliser_casse(row.get("region", "")) or None,
            commune=normaliser_casse(row.get("commune", "")) or None,
            quartier=(row.get("quartier") or "").strip() or None,
            prix=prix,
            surface=surface,
            chambres=parser_int(row.get("pieces")),
            salles_bain=parser_int(row.get("sdb")),
            statut=(row.get("statut") or "").strip() or None,
            description=(row.get("description") or "").strip() or None,
        )
        if motif_prix:
            bien.avertissements.append(f"prix : {motif_prix}")
        if motif_date:
            bien.avertissements.append(f"date : {motif_date}")
        biens.append(bien)
    return biens, erreurs


def charger_agence(chemin):
    biens, erreurs = [], []
    wb = openpyxl.load_workbook(chemin, data_only=True)
    if "Annonces" not in wb.sheetnames:
        return biens, ["feuille 'Annonces' absente"]
    ws = wb["Annonces"]
    lignes = list(ws.iter_rows(values_only=True))
    if not lignes:
        return biens, ["feuille 'Annonces' vide"]

    headers = [str(h).strip() if h is not None else "" for h in lignes[0]]
    for i, row in enumerate(lignes[1:], start=2):
        if not any(row):
            continue  # ligne vide
        data = dict(zip(headers, row))
        identifiant = data.get("Code")
        if not identifiant:
            erreurs.append(f"ligne {i} : identifiant absent")
            continue

        prix, motif_prix = parser_prix(data.get("Prix_FCFA"))
        surface, _ = parser_prix(data.get("Surface_m2"))
        # Date peut être déjà un datetime (openpyxl) ou une string
        date_val = data.get("Date")
        if hasattr(date_val, "year"):
            date, motif_date = date_val, None
        else:
            date, motif_date = parser_date(date_val)

        categorie = resoudre_categorie(data.get("Type", ""), SYNONYMES_CATEGORIES)

        bien = BienImmobilier(
            identifiant_local=str(identifiant),
            source="agence",
            date_mise_en_ligne=date,
            type_transaction="vente",  # l'agence ne propose que des ventes
            categorie=categorie,
            region=normaliser_casse(data.get("Region", "")) or None,
            commune=normaliser_casse(data.get("Ville", "")) or None,
            prix=prix,
            surface=surface,
            chambres=parser_int(data.get("Chambres")),
            salles_bain=parser_int(data.get("SDB")),
            statut=(data.get("Statut") or "").strip() if isinstance(data.get("Statut"), str) else data.get("Statut"),
        )
        if motif_prix:
            bien.avertissements.append(f"prix : {motif_prix}")
        if motif_date:
            bien.avertissements.append(f"date : {motif_date}")
        biens.append(bien)
    return biens, erreurs


def charger_particuliers(chemin):
    biens, erreurs = [], []
    with open(chemin, encoding="utf-8") as f:
        data = json.load(f)
    for i, item in enumerate(data):
        annonce = item.get("annonce", {})
        identifiant = annonce.get("id")
        if not identifiant:
            erreurs.append(f"item {i} : identifiant absent")
            continue
        loc = annonce.get("localisation", {})
        det = annonce.get("details", {})

        prix, motif_prix = parser_prix(det.get("prix"))
        surface, _ = parser_prix(det.get("surface"))
        date, motif_date = parser_date(annonce.get("date"))
        transaction = (det.get("transaction") or "").lower() or None
        categorie = resoudre_categorie(det.get("type", ""), SYNONYMES_CATEGORIES)

        bien = BienImmobilier(
            identifiant_local=identifiant,
            source="particuliers",
            date_mise_en_ligne=date,
            type_transaction=transaction,
            periodicite="mensuel" if transaction == "location" else None,
            categorie=categorie,
            region=normaliser_casse(loc.get("region", "")) or None,
            commune=normaliser_casse(loc.get("ville", "")) or None,
            prix=prix,
            surface=surface,
            chambres=parser_int(det.get("chambres")),
            salles_bain=parser_int(det.get("sdb")),
            statut=(annonce.get("statut") or "").strip() or None,
        )
        if motif_prix:
            bien.avertissements.append(f"prix : {motif_prix}")
        if motif_date:
            bien.avertissements.append(f"date : {motif_date}")
        biens.append(bien)
    return biens, erreurs


def charger_notaires(chemin):
    biens, erreurs = [], []
    contenu, enc = lire_csv_avec_fallback(chemin)
    reader = csv.DictReader(io.StringIO(contenu), delimiter=",")
    for i, row in enumerate(reader, start=1):
        identifiant = (row.get("id_acte") or "").strip()
        if not identifiant:
            erreurs.append(f"ligne {i} : identifiant absent")
            continue

        prix, motif_prix = parser_prix(row.get("montant_fcfa"))
        surface, _ = parser_prix(row.get("superficie"))
        date, motif_date = parser_date(row.get("date_acte"))

        # Uniformiser les statuts notaires
        statut_brut = (row.get("statut") or "").strip()
        statut_map = {
            "vendu": "Vendu", "vente conclue": "Vendu", "acte signé": "Vendu",
            "acte signe": "Vendu",
        }
        statut = statut_map.get(statut_brut.lower(), statut_brut)

        categorie = resoudre_categorie(row.get("type_bien", ""), SYNONYMES_CATEGORIES)

        bien = BienImmobilier(
            identifiant_local=identifiant,
            source="notaires",
            date_mise_en_ligne=date,
            type_transaction="vente",  # actes notariés = ventes
            categorie=categorie,
            region=normaliser_casse(row.get("region", "")) or None,
            commune=normaliser_casse(row.get("ville", "")) or None,
            prix=prix,
            surface=surface,
            chambres=parser_int(row.get("nb_chambres")),
            statut=statut,
        )
        if motif_prix:
            bien.avertissements.append(f"prix : {motif_prix}")
        if motif_date:
            bien.avertissements.append(f"date : {motif_date}")
        biens.append(bien)
    return biens, erreurs


def charger_historique(chemin):
    biens, erreurs = [], []
    contenu, enc = lire_csv_avec_fallback(chemin)
    reader = csv.DictReader(io.StringIO(contenu), delimiter=";")

    regions_valides = set(REFERENTIEL_GEO.keys())

    for i, row in enumerate(reader, start=1):
        identifiant = (row.get("ID") or "").strip()
        if not identifiant:
            erreurs.append(f"ligne {i} : identifiant absent")
            continue

        # Prix : N/A ou numérique avec espaces
        prix_brut = (row.get("Prix") or "").strip()
        if prix_brut.upper() == "N/A":
            prix, motif_prix = None, "prix N/A"
        else:
            prix, motif_prix = parser_prix(prix_brut)
        surface, _ = parser_prix(row.get("Surface"))
        date, motif_date = parser_date(row.get("DateVente"))

        # Résolution région/ville : d'abord corriger_mojibake, puis fuzzy matching
        region_brute = row.get("Region", "") or ""
        ville_brute = row.get("Ville", "") or ""

        region_corrigee = corriger_mojibake(region_brute)
        region = resoudre_par_referentiel(region_corrigee, regions_valides, seuil=0.75)
        if region is None and region_corrigee:
            bien.avertissements.append(f"région non résolue : '{region_brute}'")
        region = region or (normaliser_casse(region_corrigee) if region_corrigee else None)

        villes_valides = REFERENTIEL_GEO.get(region, set()) if region else set()
        ville_corrigee = corriger_mojibake(ville_brute)
        ville = resoudre_par_referentiel(ville_corrigee, villes_valides, seuil=0.75)
        if ville is None and ville_corrigee and villes_valides:
            # chercher dans toutes les villes du référentiel si on n'a pas matché dans la région
            toutes_villes = set().union(*REFERENTIEL_GEO.values())
            ville = resoudre_par_referentiel(ville_corrigee, toutes_villes, seuil=0.75)
        ville = ville or (normaliser_casse(ville_corrigee) if ville_corrigee else None)

        categorie = resoudre_categorie(row.get("TypeBien", ""), SYNONYMES_CATEGORIES)

        bien = BienImmobilier(
            identifiant_local=identifiant,
            source="historique",
            date_mise_en_ligne=date,
            type_transaction="vente",
            categorie=categorie,
            region=region,
            commune=ville,
            prix=prix,
            surface=surface,
            chambres=parser_int(row.get("Chambres")),
            salles_bain=parser_int(row.get("SallesBain")),
        )
        if motif_prix:
            bien.avertissements.append(f"prix : {motif_prix}")
        if motif_date:
            bien.avertissements.append(f"date : {motif_date}")
        biens.append(bien)
    return biens, erreurs


# =============================================================
# CHARGEMENT

chargeurs = {
    "plateforme":   ("sources/plateforme_immo_sn.csv",  charger_plateforme),
    "agence":       ("sources/agence_teranga.xlsx",     charger_agence),
    "particuliers": ("sources/particuliers.json",       charger_particuliers),
    "notaires":     ("sources/notaires.csv",            charger_notaires),
    "historique":   ("sources/import_historique.csv",   charger_historique),
}

for source, (chemin, chargeur) in chargeurs.items():
    if not os.path.exists(chemin):
        print(f"ERREUR : fichier introuvable {chemin}")
        continue
    biens, erreurs = chargeur(chemin)
    biens_bruts.extend(biens)
    inventaire_initial[source] = {
        "lignes_lues": len(biens) + len(erreurs),
        "lignes_retenues": len(biens),
        "erreurs_parsing": len(erreurs),
        "exemples_erreurs": erreurs[:3],
    }

# =============================================================
# AFFICHAGE

print("Inventaire initial :")
for source, info in inventaire_initial.items():
    print(f"  {source:15s} : {info['lignes_retenues']:4d} retenues / {info['lignes_lues']:4d} lues")
    if info["erreurs_parsing"]:
        print(f"                    {info['erreurs_parsing']} erreurs (ex: {info['exemples_erreurs'][0] if info['exemples_erreurs'] else '—'})")
print(f"\nTotal : {len(biens_bruts)} biens bruts chargés")

resoudre_categorie définie.
Inventaire initial :
  plateforme      :    0 retenues /  875 lues
                    875 erreurs (ex: ligne 1 : identifiant absent)
  agence          :  640 retenues /  640 lues
  particuliers    :  731 retenues /  731 lues
  notaires        :  410 retenues /  410 lues
  historique      :  528 retenues /  528 lues

Total : 2309 biens bruts chargés


In [26]:
import io
import csv

def charger_plateforme(chemin):
    """
    Version robuste : gère l'absence d'en-tête en ligne 1.
    - Cherche une ligne d'en-tête ('ref_plateforme' en 1re colonne), sinon utilise les noms canoniques du CDC.
    - Filtre les lignes qui ne commencent pas par 'PLT-'.
    - Normalise 'vent' -> 'vente'.
    """
    biens, erreurs = [], []
    contenu, enc = lire_csv_avec_fallback(chemin)
    lignes = list(csv.reader(io.StringIO(contenu), delimiter=";"))

    # Noms canoniques attendus (CDC : 15 colonnes)
    colonnes_attendues = [
        "ref_plateforme", "date_mise_en_ligne", "type_transaction", "categorie",
        "region", "commune", "quartier", "prix", "devise", "surface",
        "pieces", "sdb", "description", "statut", "date_expiration"
    ]

    # Chercher un éventuel en-tête existant
    index_entete = None
    for i, row in enumerate(lignes):
        if row and row[0].strip() == "ref_plateforme":
            index_entete = i
            break

    if index_entete is not None:
        colonnes = [c.strip() for c in lignes[index_entete]]
        lignes = lignes[:index_entete] + lignes[index_entete + 1:]
    else:
        colonnes = colonnes_attendues

    # Mapping de normalisation du type de transaction
    trans_map = {"vent": "vente", "vente": "vente", "location": "location",
                 "louer": "location", "loué": "location", "loue": "location"}

    for i, row in enumerate(lignes, start=1):
        if not row or len(row) < 3:
            continue
        if row[0].strip() == "ref_plateforme":  # skip header résiduel
            continue
        identifiant = row[0].strip()
        if not identifiant.startswith("PLT-"):
            continue

        data = dict(zip(colonnes, row))

        prix, motif_prix = parser_prix(data.get("prix", ""))
        surface, _ = parser_prix(data.get("surface", ""))
        date, motif_date = parser_date(data.get("date_mise_en_ligne"))

        categorie = resoudre_categorie(data.get("categorie", ""), SYNONYMES_CATEGORIES)

        trans_brute = (data.get("type_transaction") or "").strip().lower()
        type_transaction = trans_map.get(trans_brute, trans_brute) or None

        devise = (data.get("devise") or "").lower()
        periodicite = "mensuel" if "/mois" in devise else None

        bien = BienImmobilier(
            identifiant_local=identifiant,
            source="plateforme",
            date_mise_en_ligne=date,
            type_transaction=type_transaction,
            periodicite=periodicite,
            categorie=categorie,
            region=normaliser_casse(data.get("region", "")) or None,
            commune=normaliser_casse(data.get("commune", "")) or None,
            quartier=(data.get("quartier") or "").strip() or None,
            prix=prix,
            surface=surface,
            chambres=parser_int(data.get("pieces")),
            salles_bain=parser_int(data.get("sdb")),
            statut=(data.get("statut") or "").strip() or None,
            description=(data.get("description") or "").strip() or None,
        )
        if motif_prix:
            bien.avertissements.append(f"prix : {motif_prix}")
        if motif_date:
            bien.avertissements.append(f"date : {motif_date}")
        biens.append(bien)
    return biens, erreurs


# Recharger plateforme
biens_plateforme, erreurs_plateforme = charger_plateforme("sources/plateforme_immo_sn.csv")
print(f"Plateforme : {len(biens_plateforme)} biens chargés, {len(erreurs_plateforme)} erreurs")

# Retirer l'ancienne plateforme (au cas où) et fusionner
biens_bruts = [b for b in biens_bruts if b.source != "plateforme"]
biens_bruts.extend(biens_plateforme)

# Mettre à jour l'inventaire
inventaire_initial["plateforme"] = {
    "lignes_lues": len(biens_plateforme) + len(erreurs_plateforme),
    "lignes_retenues": len(biens_plateforme),
    "erreurs_parsing": len(erreurs_plateforme),
    "exemples_erreurs": erreurs_plateforme[:3],
}

# Vérification
from collections import Counter
print(f"\nTotal biens bruts : {len(biens_bruts)}")
print("Répartition par source :")
for s, n in Counter(b.source for b in biens_bruts).items():
    print(f"  {s:15s} : {n}")

Plateforme : 875 biens chargés, 0 erreurs

Total biens bruts : 3184
Répartition par source :
  agence          : 640
  particuliers    : 731
  notaires        : 410
  historique      : 528
  plateforme      : 875


# Étape 4 — Validation, normalisation et quarantaine

## Ce que fait cette cellule

Elle applique les règles métier du CDC section 6 à tous les biens bruts, et les classe dans trois catégories exclusives :

| Catégorie | Condition | Destination |
|---|---|---|
| **Propre** | Toutes les règles bloquantes passent, aucun avertissement | `donnees_propres.csv` |
| **Avertissement** | Règles bloquantes OK, mais au moins une règle d'avertissement échoue | `avertissements.csv` |
| **Quarantaine** | Au moins une règle bloquante échoue | `quarantine.csv` avec motif |

Elle calcule aussi les **indicateurs de qualité obligatoires** du CDC section 7.

## Règles bloquantes appliquées (CDC 6.1)

- Identifiant local absent
- `type_transaction` différent de `vente` ou `location`
- `prix` absent ou non strictement positif
- `surface` absente ou non strictement positive
- `date_mise_en_ligne` non interprétable ou future
- `categorie` hors des 7 catégories du référentiel

## Règles d'avertissement appliquées (CDC 6.2)

- `surface` < 5 m² ou > 2 000 m²
- `prix` (vente) < 100 000 FCFA ou > 2 000 000 000 FCFA
- `loyer` (location) < 25 000 FCFA ou > 5 000 000 FCFA
- `chambres` > 10 pour un appartement ou un studio
- `salles_bain` > `chambres` + 2
- Région inconnue du référentiel géographique
- Incohérence région/ville (la ville n'appartient pas à la région dans le référentiel)

## Détection des doublons exacts

Deux biens sont doublons exacts si les 15 champs métier (source + identifiant + date + caractéristiques) sont identiques. Les doublons sont déplacés en quarantaine avec motif « doublon exact ».

## Indicateurs de qualité produits (CDC section 7)

- Nombre total d'enregistrements reçus
- Nombre de valeurs manquantes par colonne
- Nombre et taux de doublons exacts
- Nombre d'erreurs de conversion par champ
- Nombre en quarantaine
- Nombre conservés avec avertissement
- Nombre entièrement conformes
- Taux global de conformité

## Justifications des choix

| Choix | Justification CDC |
|---|---|
| 3 catégories exclusives | Section 6.1 : bloquante vs normalisable vs avertissement |
| Cohérence géo en avertissement, pas en quarantaine | Section 6.1 : « signaler, ne pas corriger sans preuve » |
| Doublons exacts en quarantaine | Section 7 : les doublons doivent être comptés, pas supprimés silencieusement |
| Seuils vente et location séparés | Section 6.2 : « loyer par mois, seuils à ajuster selon la périodicité » |
| Catégorie comparée au référentiel | Section 6.1 : « Quarantaine si hors référentiel » |

## Ce que tu dois voir à la fin


In [ ]:
from collections import Counter
from datetime import datetime
import os

os.chdir("/kaggle/working")

# =============================================================
# 1. RÈGLES MÉTIER (CDC section 6)

CATEGORIES_VALIDES = set(SYNONYMES_CATEGORIES.values())


def valider_bien(bien):
    motifs_bloquants = []
    motifs_avertissement = []

    if not bien.identifiant_local:
        motifs_bloquants.append("identifiant local absent")
    if bien.type_transaction not in ("vente", "location"):
        motifs_bloquants.append(f"type_transaction invalide : {bien.type_transaction}")
    if bien.prix is None:
        motifs_bloquants.append("prix manquant")
    elif bien.prix <= 0:
        motifs_bloquants.append(f"prix non strictement positif : {bien.prix}")
    if bien.surface is None:
        motifs_bloquants.append("surface manquante")
    elif bien.surface <= 0:
        motifs_bloquants.append(f"surface non strictement positive : {bien.surface}")
    if bien.date_mise_en_ligne is None:
        motifs_bloquants.append("date non interprétable")
    elif bien.date_mise_en_ligne > datetime.now():
        motifs_bloquants.append(f"date future : {bien.date_mise_en_ligne.date()}")
    if bien.categorie not in CATEGORIES_VALIDES:
        motifs_bloquants.append(f"catégorie hors référentiel : {bien.categorie}")

    if bien.surface is not None and (bien.surface < 5 or bien.surface > 2000):
        motifs_avertissement.append(f"surface atypique : {bien.surface} m²")

    if bien.prix is not None:
        if bien.type_transaction == "vente":
            if bien.prix < 100_000 or bien.prix > 2_000_000_000:
                motifs_avertissement.append(f"prix de vente atypique : {bien.prix} FCFA")
        elif bien.type_transaction == "location":
            if bien.prix < 25_000 or bien.prix > 5_000_000:
                motifs_avertissement.append(f"loyer atypique : {bien.prix} FCFA/mois")

    if (bien.chambres is not None and bien.chambres > 10
            and bien.categorie in ("Appartement", "Studio")):
        motifs_avertissement.append(
            f"nombre de chambres atypique pour {bien.categorie} : {bien.chambres}"
        )
    if (bien.chambres is not None and bien.salles_bain is not None
            and bien.salles_bain > bien.chambres + 2):
        motifs_avertissement.append(
            f"salles de bain > chambres + 2 : {bien.salles_bain} vs {bien.chambres}"
        )

    if bien.region and bien.commune:
        if bien.region not in REFERENTIEL_GEO:
            motifs_avertissement.append(f"région inconnue du référentiel : {bien.region}")
        elif bien.commune not in REFERENTIEL_GEO[bien.region]:
            motifs_avertissement.append(
                f"incohérence région/ville : {bien.region} / {bien.commune}"
            )

    if motifs_bloquants:
        bien.motif_quarantaine = "; ".join(motifs_bloquants)
        bien.avertissements.extend(motifs_avertissement)
        return "quarantaine", motifs_bloquants
    elif motifs_avertissement or bien.avertissements:
        bien.avertissements.extend(motifs_avertissement)
        return "avertissement", bien.avertissements
    else:
        return "propre", []


# =============================================================
# 2. DÉDOUBLONNAGE AVANT VALIDATION (correction du bug)

def cle_doublon(bien):
    return tuple(getattr(bien, c) for c in BienImmobilier.CHAMPS)


vus = set()
biens_uniques = []
doublons_exacts = []
biens_quarantaine = []  # initialisé ici, accueillera aussi les doublons

for bien in biens_bruts:
    k = cle_doublon(bien)
    if k in vus:
        bien.motif_quarantaine = "doublon exact"
        biens_quarantaine.append(bien)
        doublons_exacts.append(bien)
    else:
        vus.add(k)
        biens_uniques.append(bien)

print(f"Dédoublonnage : {len(doublons_exacts)} doublons exacts détectés "
      f"sur {len(biens_bruts)} biens bruts "
      f"({len(biens_uniques)} uniques)")

# =============================================================
# 3. VALIDATION DES BIENS UNIQUES

biens_propres = []
biens_avertissement = []

for bien in biens_uniques:
    statut, motifs = valider_bien(bien)
    if statut == "quarantaine":
        biens_quarantaine.append(bien)
    elif statut == "avertissement":
        biens_avertissement.append(bien)
    else:
        biens_propres.append(bien)

# =============================================================
# 4. INDICATEURS DE QUALITÉ (CDC section 7)

def compter_manquants(biens, champ):
    return sum(1 for b in biens if getattr(b, champ) in (None, "", []))


total = len(biens_bruts)

manquants_par_colonne = {
    champ: compter_manquants(biens_bruts, champ)
    for champ in BienImmobilier.CHAMPS
}

erreurs_conversion = Counter()
for b in biens_bruts:
    for av in b.avertissements:
        if av.startswith("prix :"):
            erreurs_conversion["prix"] += 1
        elif av.startswith("date :"):
            erreurs_conversion["date"] += 1

avertissements_par_motif = Counter()
for b in biens_avertissement:
    for av in b.avertissements:
        motif = av.split(":")[0].strip()
        avertissements_par_motif[motif] += 1

quarantaine_par_motif = Counter()
for b in biens_quarantaine:
    if b.motif_quarantaine:
        for m in b.motif_quarantaine.split(";"):
            motif = m.split(":")[0].strip()
            quarantaine_par_motif[motif] += 1

indicateurs_qualite = {
    "total_recus": total,
    "conformes": len(biens_propres),
    "avec_avertissement": len(biens_avertissement),
    "en_quarantaine": len(biens_quarantaine),
    "doublons_exacts": len(doublons_exacts),
    "taux_conformite_pct": round(100 * len(biens_propres) / total, 2) if total else 0.0,
    "taux_quarantaine_pct": round(100 * len(biens_quarantaine) / total, 2) if total else 0.0,
    "taux_avertissement_pct": round(100 * len(biens_avertissement) / total, 2) if total else 0.0,
    "repartition_initiale_par_source": {
        s: info["lignes_retenues"] for s, info in inventaire_initial.items()
    },
    "valeurs_manquantes_par_colonne": manquants_par_colonne,
    "erreurs_conversion_par_champ": dict(erreurs_conversion),
    "avertissements_par_motif": dict(avertissements_par_motif),
    "quarantaine_par_motif": dict(quarantaine_par_motif),
}

# =============================================================
# 5. AFFICHAGE

print("\n=== Rapport de qualité (étape 4) ===\n")
for k in ["total_recus", "conformes", "avec_avertissement", "en_quarantaine",
          "doublons_exacts", "taux_conformite_pct", "taux_quarantaine_pct",
          "taux_avertissement_pct"]:
    print(f"  {k:30s} : {indicateurs_qualite[k]}")

print("\n--- Valeurs manquantes par colonne ---")
for champ, n in indicateurs_qualite["valeurs_manquantes_par_colonne"].items():
    if n > 0:
        print(f"  {champ:25s} : {n:4d} manquants")

print("\n--- Erreurs de conversion par champ ---")
for champ, n in indicateurs_qualite["erreurs_conversion_par_champ"].items():
    print(f"  {champ:25s} : {n:4d} erreurs")

print("\n--- Avertissements par motif ---")
for motif, n in sorted(indicateurs_qualite["avertissements_par_motif"].items(),
                        key=lambda x: -x[1]):
    print(f"  {motif:45s} : {n:4d}")

print("\n--- Quarantaine par motif ---")
for motif, n in sorted(indicateurs_qualite["quarantaine_par_motif"].items(),
                        key=lambda x: -x[1]):
    print(f"  {motif:45s} : {n:4d}")

Dédoublonnage : 15 doublons exacts détectés sur 3184 biens bruts (3169 uniques)

=== Rapport de qualité (étape 4) ===

  total_recus                    : 3184
  conformes                      : 2811
  avec_avertissement             : 168
  en_quarantaine                 : 205
  doublons_exacts                : 15
  taux_conformite_pct            : 88.29
  taux_quarantaine_pct           : 6.44
  taux_avertissement_pct         : 5.28

--- Valeurs manquantes par colonne ---
  date_mise_en_ligne        :    6 manquants
  periodicite               : 2788 manquants
  commune                   :    8 manquants
  prix                      :   93 manquants
  surface                   :   84 manquants
  salles_bain               :  410 manquants
  statut                    :  528 manquants
  quartier                  : 2309 manquants
  description               : 2474 manquants

--- Erreurs de conversion par champ ---
  prix                      :   93 erreurs
  date                      :    6 

# Étape 5 — Statistiques descriptives

## Ce que fait cette cellule

Elle calcule les statistiques demandées par l'étape 5 du CDC :
- Prix médians (ventes et locations séparément)
- Statistiques par type de bien
- Prix au m² par type (quand surface > 0)
- Statistiques par ville et par source

Aucune bibliothèque externe : uniquement `statistics` (stdlib).

## Périmètre d'analyse

`corpus_analyse = biens_propres + biens_avertissement` = 2 979 biens.

Pourquoi inclure les avertissements ?
CDC 6.1 : « avertissement = conserver et signaler ». Ce ne sont pas des erreurs, juste des valeurs atypiques. Les exclure reviendrait à biaiser les distributions.

Les biens en quarantaine (205) sont exclus : ils sont inutilisables.

## Séparation ventes / locations (CDC 5.4)

**Critique** : un loyer mensuel de 250 000 FCFA n'est pas comparable à un prix de vente de 250 000 FCFA. Mélanger les deux donne une médiane absurde.

On calcule donc :
- Prix médian **ventes** uniquement
- Loyer médian **locations** uniquement, avec détail par périodicité

## Prix au m²

Calculé uniquement quand `prix > 0` et `surface > 0` et `surface >= 5` (les surfaces < 5 m² sont atypiques, cf CDC 6.2).

## Statistiques par source

Chaque source a ses propres biais (une agence ne vend pas les mêmes biens que la plateforme, les notaires enregistrent des ventes conclues, etc.). Séparer permet de repérer une source qui écrase les autres.

## Ce que tu dois voir à la fin

\`\`\`
=== Stats globales ===
  Ventes      : n=XXXX, prix médian=XX M FCFA, moyenne=XX M FCFA
  Locations   : n=XXXX, loyer médian=XXX XXX FCFA

=== Prix médian de vente par type ===
  Appartement : XX M FCFA (n=XXX)
  Villa       : XXX M FCFA (n=XXX)
  ...

=== Prix au m² par type (ventes) ===
  Appartement : XXX XXX FCFA/m² (n=XXX)
  Villa       : XXX XXX FCFA/m² (n=XXX)
  ...

=== Top 10 communes par nombre de ventes ===
  Dakar       : XXX ventes, prix médian = XX M FCFA
  ...

=== Stats par source (ventes) ===
  agence      : XXX ventes, prix médian = XX M FCFA
  plateforme  : XXX ventes, prix médian = XX M FCFA
  ...

=== Stats locations par périodicité ===
  mensuel     : XXX locations, loyer médian = XXX XXX FCFA
  ...
\`\`\`

## Ce qu'il faut vérifier dans les résultats

| Vérification | Signe que tout va bien |
|---|---|
| Prix médian Villa > prix médian Appartement | Cohérent avec le marché |
| Prix médian Terrain > prix médian Studio (à surface égale) | Cohérent |
| Prix médian par source comparable | Sinon, biais de source à investiguer |
| Prix au m² < 5 000 000 FCFA/m² | Au-delà, parsing suspect |
| Loyer médian < 5 000 000 FCFA | Au-delà, confondre vente/location |

## Limites assumées

- **Rapprochement inter-sources non fait** : le même bien peut apparaître dans plusieurs sources et être compté plusieurs fois. Cf extension axe 1.
- **Prix demandés, pas prix conclus** : pour les annonces (plateforme, particuliers, agence), ce sont des prix demandés. Pour les notaires, ce sont des actes signés (plus proches du prix réel).
- **Coordonnées non utilisées** : le référentiel géographique fournit des coordonnées, mais on ne les utilise pas pour calculer des distances (hors périmètre).

In [ ]:
import statistics
from collections import defaultdict, Counter
import os

os.chdir("/kaggle/working")

# =============================================================
# 1. CORPUS D'ANALYSE

corpus_analyse = biens_propres + biens_avertissement
ventes = [b for b in corpus_analyse if b.type_transaction == "vente"]
locations = [b for b in corpus_analyse if b.type_transaction == "location"]

print(f"Corpus analysable : {len(corpus_analyse)} biens")
print(f"  Dont ventes     : {len(ventes)}")
print(f"  Dont locations  : {len(locations)}")


# =============================================================
# 2. HELPERS

def stats(lst):
    """Retourne (n, moyenne, médiane) en ignorant les None."""
    vals = [x for x in lst if x is not None]
    if not vals:
        return 0, None, None
    return (
        len(vals),
        round(statistics.mean(vals), 2),
        round(statistics.median(vals), 2),
    )


def fmt_fcfa(v):
    """Formate un montant FCFA en M FCFA si grand."""
    if v is None:
        return "—"
    if v >= 1_000_000:
        return f"{v/1_000_000:.1f} M FCFA"
    return f"{v:,.0f} FCFA"


# =============================================================
# 3. STATS GLOBALES

print("\n=== Stats globales ===")
n_v, mean_v, med_v = stats([b.prix for b in ventes])
n_l, mean_l, med_l = stats([b.prix for b in locations])

print(f"  Ventes     : n={n_v:4d}  |  prix médian = {fmt_fcfa(med_v)}  |  moyenne = {fmt_fcfa(mean_v)}")
print(f"  Locations  : n={n_l:4d}  |  loyer médian = {fmt_fcfa(med_l)}  |  moyenne = {fmt_fcfa(mean_l)}")

n_vs, _, med_vs = stats([b.surface for b in ventes])
n_ls, _, med_ls = stats([b.surface for b in locations])
print(f"\n  Surfaces   : ventes médiane = {med_vs} m² (n={n_vs})  |  locations médiane = {med_ls} m² (n={n_ls})")


# =============================================================
# 4. STATS PAR CATÉGORIE (ventes)

print("\n=== Prix médian de vente par type de bien ===")
prix_par_cat = defaultdict(list)
for b in ventes:
    if b.categorie and b.prix:
        prix_par_cat[b.categorie].append(b.prix)

resultats_par_cat = {}
for cat in sorted(prix_par_cat):
    n, mean, med = stats(prix_par_cat[cat])
    resultats_par_cat[cat] = {"n": n, "mean": mean, "median": med}
    print(f"  {cat:20s} : n={n:4d}  |  médiane = {fmt_fcfa(med)}")


# =============================================================
# 5. PRIX AU M² PAR CATÉGORIE (ventes)

print("\n=== Prix au m² par type (ventes, surface >= 5) ===")
prix_m2_par_cat = defaultdict(list)
for b in ventes:
    if b.categorie and b.prix and b.surface and b.surface >= 5:
        prix_m2_par_cat[b.categorie].append(b.prix / b.surface)

resultats_m2 = {}
for cat in sorted(prix_m2_par_cat):
    n, mean, med = stats(prix_m2_par_cat[cat])
    resultats_m2[cat] = {"n": n, "mean": mean, "median": med}
    print(f"  {cat:20s} : n={n:4d}  |  médiane = {fmt_fcfa(med)}/m²")


# =============================================================
# 6. TOP COMMUNES PAR NOMBRE DE VENTES

print("\n=== Top 10 communes par nombre de ventes ===")
ventes_par_commune = defaultdict(list)
for b in ventes:
    if b.commune and b.prix:
        ventes_par_commune[b.commune].append(b.prix)

top_communes = sorted(ventes_par_commune.items(), key=lambda x: -len(x[1]))[:10]
for commune, prix_list in top_communes:
    n, mean, med = stats(prix_list)
    print(f"  {commune:25s} : n={n:4d}  |  médiane = {fmt_fcfa(med)}")


# =============================================================
# 7. STATS PAR SOURCE

print("\n=== Stats par source (ventes) ===")
stats_source = defaultdict(lambda: {"ventes": [], "locations": []})
for b in corpus_analyse:
    if b.prix:
        if b.type_transaction == "vente":
            stats_source[b.source]["ventes"].append(b.prix)
        elif b.type_transaction == "location":
            stats_source[b.source]["locations"].append(b.prix)

resultats_source = {}
for source in sorted(stats_source):
    v = stats_source[source]["ventes"]
    l = stats_source[source]["locations"]
    n_v, _, med_v = stats(v)
    n_l, _, med_l = stats(l)
    resultats_source[source] = {
        "ventes_n": n_v, "ventes_median": med_v,
        "locations_n": n_l, "locations_median": med_l,
    }
    print(f"  {source:15s} : ventes n={n_v:4d} (médiane={fmt_fcfa(med_v)})  |  "
          f"locations n={n_l:4d} (médiane={fmt_fcfa(med_l)})")


# =============================================================
# 8. STATS LOCATIONS PAR PÉRIODICITÉ

print("\n=== Locations par périodicité ===")
loyers_par_periodicite = defaultdict(list)
for b in locations:
    if b.prix:
        per = b.periodicite or "non précisé"
        loyers_par_periodicite[per].append(b.prix)

resultats_periodicite = {}
for per in sorted(loyers_par_periodicite):
    n, mean, med = stats(loyers_par_periodicite[per])
    resultats_periodicite[per] = {"n": n, "mean": mean, "median": med}
    print(f"  {per:15s} : n={n:4d}  |  loyer médian = {fmt_fcfa(med)}")


# =============================================================
# 9. RÉSUMÉ STRUCTURÉ (pour cellule 6)

stats_globales = {
    "ventes": {"n": n_v, "mean": mean_v, "median": med_v},
    "locations": {"n": n_l, "mean": mean_l, "median": med_l},
    "surfaces": {"ventes_median": med_vs, "locations_median": med_ls},
}

stats_par_type = resultats_par_cat
prix_m2_par_type = resultats_m2
stats_par_source = resultats_source
stats_par_periodicite = resultats_periodicite
stats_par_ville = {
    commune: stats(prix_list) for commune, prix_list in ventes_par_commune.items()
}

print("\nCellule 5 terminée. Objets `stats_globales`, `stats_par_type`, "
      "`prix_m2_par_type`, `stats_par_source`, `stats_par_periode`, "
      "`stats_par_ville` prêts pour l'export (cellule 6).")

Corpus analysable : 2979 biens
  Dont ventes     : 2636
  Dont locations  : 343

=== Stats globales ===
  Ventes     : n=2636  |  prix médian = 71.4 M FCFA  |  moyenne = 100.4 M FCFA
  Locations  : n= 343  |  loyer médian = 320,000 FCFA  |  moyenne = 406,997 FCFA

  Surfaces   : ventes médiane = 146.95 m² (n=2636)  |  locations médiane = 149.8 m² (n=343)

=== Prix médian de vente par type de bien ===
  Appartement          : n= 609  |  médiane = 68.6 M FCFA
  Chambre              : n= 163  |  médiane = 11.7 M FCFA
  Local commercial     : n= 145  |  médiane = 101.5 M FCFA
  Maison               : n= 559  |  médiane = 100.1 M FCFA
  Studio               : n= 317  |  médiane = 28.3 M FCFA
  Terrain              : n= 372  |  médiane = 47.9 M FCFA
  Villa                : n= 471  |  médiane = 253.6 M FCFA

=== Prix au m² par type (ventes, surface >= 5) ===
  Appartement          : n= 609  |  médiane = 643,658 FCFA/m²
  Chambre              : n= 163  |  médiane = 562,577 FCFA/m²
  Local com

# Étape 7 — Export et reproductibilité

## Ce que fait cette cellule

Elle exporte tous les livrables du CDC :

| Fichier | Contenu | Format |
|---|---|---|
| `sortie/donnees_propres.csv` | Biens conformes | CSV |
| `sortie/avertissements.csv` | Biens conservés avec signalement | CSV |
| `sortie/quarantine.csv` | Biens rejetés + motif | CSV |
| `sortie/rapport_qualite.json` | Tous les indicateurs + stats | JSON |
| `sortie/rapport_qualite.csv` | Indicateurs globaux (lisible) | CSV |
| `sortie/donnees_propres.json` | Version JSON des biens propres | JSON |
| `sortie/README.md` | Documentation | Markdown |

Conformément au CDC section 11 (livrables 3, 4, 5, 7) et section 5.8 (exports CSV et JSON).

## Justifications des choix

| Choix | Justification CDC |
|---|---|
| 3 CSV distincts (propres/avertissements/quarantaine) | Section 11 : distinguer les 3 catégories |
| JSON **et** CSV pour le rapport | Section 11 : exports lisibles et cohérents |
| JSON pour `donnees_propres` | Section 5.8 : « exports CSV et JSON » |
| README détaillé | Section 11 (livrable 9) : installation, exécution, règles, résultats |
| Sérialisation des `datetime` en ISO | Lecture universelle, pas d'ambiguïté |

## Ce que tu dois voir à la fin

\`\`\`
Fichiers produits dans sortie/ :
  donnees_propres.csv       :  XXX 000 octets
  donnees_propres.json      :  XXX 000 octets
  avertissements.csv        :   XX 000 octets
  quarantine.csv            :   XX 000 octets
  rapport_qualite.json      :    XX 000 octets
  rapport_qualite.csv       :     X 000 octets
  README.md                 :     X 000 octets

Pour récupérer : onglet 'Output' à droite → dossier 'sortie/'
\`\`\`

## Comment récupérer les fichiers sur Kaggle

Après exécution, dans le panneau de droite, onglet **Output**, tu verras le dossier `sortie/`. Clic droit sur chaque fichier → **Download**.

## Limites assumées (dans le README)

- Rapprochement inter-sources non implémenté (hors périmètre, extension axe 1)
- Prix demandés (annonces), pas prix conclus (sauf notaires)
- Coordonnées GPS non utilisées
- Périodicités autres que mensuelle non distinguées

In [ ]:
import os
import csv
import json

os.chdir("/kaggle/working")
os.makedirs("sortie", exist_ok=True)


# =============================================================
# 1. EXPORT CSV (3 fichiers)

def exporter_csv(biens, chemin):
    """Export CSV UTF-8 avec tous les champs métier + motif."""
    champs = BienImmobilier.CHAMPS + ["motif_quarantaine", "avertissements"]
    with open(chemin, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=champs)
        writer.writeheader()
        for b in biens:
            row = b.to_dict()
            row["motif_quarantaine"] = b.motif_quarantaine or ""
            row["avertissements"] = "; ".join(b.avertissements)
            if row["date_mise_en_ligne"]:
                row["date_mise_en_ligne"] = row["date_mise_en_ligne"].isoformat()
            writer.writerow(row)


exporter_csv(biens_propres, "sortie/donnees_propres.csv")
exporter_csv(biens_avertissement, "sortie/avertissements.csv")
exporter_csv(biens_quarantaine, "sortie/quarantine.csv")

print(f"donnees_propres.csv  : {len(biens_propres)} lignes")
print(f"avertissements.csv   : {len(biens_avertissement)} lignes")
print(f"quarantine.csv       : {len(biens_quarantaine)} lignes")


# =============================================================
# 2. EXPORT JSON DES DONNÉES PROPRES (demandé par CDC 5.8)

def bien_vers_dict_serialisable(b):
    """Convertit un bien en dict JSON-sérialisable."""
    d = b.to_dict()
    if d["date_mise_en_ligne"]:
        d["date_mise_en_ligne"] = d["date_mise_en_ligne"].isoformat()
    return d


with open("sortie/donnees_propres.json", "w", encoding="utf-8") as f:
    json.dump(
        [bien_vers_dict_serialisable(b) for b in biens_propres],
        f, ensure_ascii=False, indent=2,
    )
print(f"donnees_propres.json : {len(biens_propres)} objets")


# =============================================================
# 3. RAPPORT QUALITÉ JSON


rapport_complet = {
    "indicateurs_globaux": indicateurs_qualite,
    "stats_globales": stats_globales,
    "stats_par_type": stats_par_type,
    "prix_m2_par_type": prix_m2_par_type,
    "stats_par_ville": stats_par_ville,
    "stats_par_source": stats_par_source,
    "stats_par_periodicite": stats_par_periodicite,
    "inventaire_initial_par_source": inventaire_initial,
}

with open("sortie/rapport_qualite.json", "w", encoding="utf-8") as f:
    json.dump(rapport_complet, f, ensure_ascii=False, indent=2, default=str)


# =============================================================
# 4. RAPPORT QUALITÉ CSV (lisible)

with open("sortie/rapport_qualite.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["categorie", "indicateur", "valeur"])
    for k, v in indicateurs_qualite.items():
        if isinstance(v, dict):
            continue  # on éclate les dicts en lignes séparées ci-dessous
        writer.writerow(["global", k, v])
    for k, v in indicateurs_qualite.items():
        if isinstance(v, dict):
            for sous_k, sous_v in v.items():
                writer.writerow([k, sous_k, sous_v])


# =============================================================
# 5. README


readme = """# ImmoAnalyse — Pipeline de préparation des données

## Environnement
Kaggle Notebook. Dépendance externe : `openpyxl` (préinstallé).

## Installation
Aucune installation nécessaire si tu utilises Kaggle. Pour un environnement local :
    pip install openpyxl

## Structure des sources
Les 9 fichiers d'entrée (5 sources + 4 référentiels) doivent être placés dans :
- `sources/` : les 5 flux bruts
- `referentiels/` : les 4 fichiers de référence

## Exécution
Exécuter les cellules dans l'ordre :
1. Setup (cellule 1)
2. Socle Python (cellule 2)
3. Résolution des catégories (cellule 2bis)
4. Chargement (cellule 3)
5. Correction plateforme (cellule 3bis)
6. Validation et quarantaine (cellule 4)
7. Statistiques (cellule 5)
8. Export (cellule 6)

## Sorties produites
- `sortie/donnees_propres.csv` et `.json` : biens conformes
- `sortie/avertissements.csv` : biens conservés avec avertissement
- `sortie/quarantine.csv` : biens rejetés avec motif
- `sortie/rapport_qualite.json` : tous les indicateurs et stats
- `sortie/rapport_qualite.csv` : indicateurs lisibles

## Règles métier appliquées
Voir CDC section 6 :
- Bloquantes : identifiant, prix > 0, surface > 0, date interprétable et non future, catégorie valide.
- Avertissements : surface atypique, prix atypique, salles de bain > chambres + 2, incohérence géographique.
- Doublons exacts déplacés en quarantaine.

## Limites connues
- Rapprochement inter-sources non implémenté (hors périmètre obligatoire, extension axe 1).
- Prix demandés (annonces) et non prix conclus, sauf notaires (actes signés).
- Périodicités de loyer autres que mensuelle non distinguées.
- Coordonnées GPS du référentiel non utilisées.
- Pas de modèle ML ni de base de données (hors périmètre).
"""
with open("sortie/README.md", "w", encoding="utf-8") as f:
    f.write(readme)


# =============================================================
# 6. VÉRIFICATION FINALE

print("\n=== Fichiers produits dans sortie/ ===")
for f in sorted(os.listdir("sortie")):
    chemin = os.path.join("sortie", f)
    taille = os.path.getsize(chemin)
    print(f"  {f:30s} : {taille:>8} octets")

print("\n=== Résumé final ===")
print(f"  Biens bruts chargés     : {indicateurs_qualite['total_recus']}")
print(f"  Biens propres           : {indicateurs_qualite['conformes']}")
print(f"  Biens avec avertissement: {indicateurs_qualite['avec_avertissement']}")
print(f"  Biens en quarantaine    : {indicateurs_qualite['en_quarantaine']}")
print(f"  Dont doublons exacts    : {indicateurs_qualite['doublons_exacts']}")
print(f"  Taux de conformité      : {indicateurs_qualite['taux_conformite_pct']} %")

print("\nPour récupérer : onglet 'Output' à droite → dossier 'sortie/'")

donnees_propres.csv  : 2811 lignes
avertissements.csv   : 168 lignes
quarantine.csv       : 205 lignes
donnees_propres.json : 2811 objets

=== Fichiers produits dans sortie/ ===
  README.md                      :     1778 octets
  avertissements.csv             :    26666 octets
  donnees_propres.csv            :   334071 octets
  donnees_propres.json           :  1212634 octets
  quarantine.csv                 :    28916 octets
  rapport_qualite.csv            :     1751 octets
  rapport_qualite.json           :     7936 octets

=== Résumé final ===
  Biens bruts chargés     : 3184
  Biens propres           : 2811
  Biens avec avertissement: 168
  Biens en quarantaine    : 205
  Dont doublons exacts    : 15
  Taux de conformité      : 88.29 %

Pour récupérer : onglet 'Output' à droite → dossier 'sortie/'
